<a href="https://colab.research.google.com/github/danielpazrosseboe/MasterDRC/blob/main/Cluster_Distribution_in_SSA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Robust Africa map for clusters_yeh_spec.csv (handles various Natural Earth schemas)
import os, io, zipfile, requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

CLUST_PATH = "/content/drive/My Drive/Master_Thesis/Surveys/clusters_yeh_spec.csv"
OUT_PATH   = "/content/drive/My Drive/Master_Thesis/Surveys/africa_clusters.png"

# 1) Load clusters
cl = pd.read_csv(CLUST_PATH, usecols=["country","year","cluster","lat","lon"])
cl["lat"] = pd.to_numeric(cl["lat"], errors="coerce")
cl["lon"] = pd.to_numeric(cl["lon"], errors="coerce")
cl = cl.dropna(subset=["lat","lon"])

gdf = gpd.GeoDataFrame(
    cl,
    geometry=gpd.points_from_xy(cl["lon"], cl["lat"]),
    crs="EPSG:4326"
)

# 2) Ensure Natural Earth admin0 is available (via GitHub mirror)
zip_url = "https://github.com/nvkelso/natural-earth-vector/archive/refs/heads/master.zip"
local_zip = "/content/ne_data.zip"
extract_dir = "/content/ne_data"

if not os.path.exists(extract_dir):
    print("Downloading Natural Earth shapefile from GitHub...")
    r = requests.get(zip_url, timeout=60)
    r.raise_for_status()
    with open(local_zip, "wb") as f:
        f.write(r.content)
    with zipfile.ZipFile(local_zip, "r") as z:
        z.extractall(extract_dir)

# Find the admin_0 countries shapefile
shp_path = None
for root, _, files in os.walk(extract_dir):
    if "ne_110m_admin_0_countries.shp" in files:
        shp_path = os.path.join(root, "ne_110m_admin_0_countries.shp")
        break
if shp_path is None:
    raise FileNotFoundError("ne_110m_admin_0_countries.shp not found in Natural Earth archive.")

world = gpd.read_file(shp_path)

# 3) Select Africa robustly
africa = None
for col in ["CONTINENT", "continent", "CONTINENT_A", "CONTINENT_LC", "CONTINEN"]:
    if col in world.columns:
        africa = world[world[col].astype(str).str.contains("Africa", case=False, na=False)]
        break

if africa is None:
    for col in ["REGION_UN", "region_un", "REGION_WB"]:
        if col in world.columns:
            africa = world[world[col].astype(str).str.contains("Africa", case=False, na=False)]
            break

# Final fallback: spatial bounding box over Africa
if africa is None or africa.empty:
    # lon: -25..60, lat: -40..40 (rough Africa box)
    africa = world.cx[-25:60, -40:40]

# 4) Plot
fig, ax = plt.subplots(figsize=(10, 10))
africa.boundary.plot(ax=ax, linewidth=0.5)
gdf.plot(ax=ax, markersize=0.08, alpha=0.6)  # default colors per notebook policy
ax.set_title("DHS Cluster Distribution over Africa", fontsize=14)
ax.set_xlabel("Longitude", fontsize = 12)
ax.set_ylabel("Latitude", fontsize = 12)
ax.set_aspect("equal", adjustable="box")
plt.tight_layout()

Path(OUT_PATH).parent.mkdir(parents=True, exist_ok=True)
plt.savefig(OUT_PATH, dpi=220, bbox_inches="tight")
print(f"Map saved → {OUT_PATH}")


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/My Drive/Master_Thesis/Surveys/clusters_yeh_spec.csv'